# 2. First look at jets and cells

Read a small sample, check it, then plot it.

We take the first **1,000 rows per LLP file** and **6 per unpaired file**. Six is a per-file limit, not the total unpaired sample. This is not random sampling.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display, Image, Markdown

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "analysis/__init__.py").is_file())
sys.path.insert(0, str(PROJECT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT / "analysis/.cache/matplotlib"))
from analysis.data.first_look import source_inventory, extract_pilot, summarize_pilot
from analysis.evaluation.first_look_plots import make_plots

OUTPUT = PROJECT / "analysis/outputs/first_look"
inventory = source_inventory(PROJECT, OUTPUT)
display(pd.DataFrame(inventory["group_summary"])[
    ["sample", "files", "total_entries", "pilot_rows_per_file", "planned_pilot_rows"]])

## Check the rows

Do the array lengths and jet → cluster → cell counts agree? Zero failures means the check passed.

The later cell reader also checks the order when assigning cells to jets.

In [ ]:
jets, rows, cells, examples = extract_pilot(PROJECT, inventory, OUTPUT)
summary = summarize_pilot(jets, rows, cells, examples, inventory, OUTPUT)
counts = pd.DataFrame(summary["groups"])
display(counts[["sample", "files_read", "rows_read", "jets", "repeated_event_keys"]])
checks = [column for column in counts if column.endswith("_failures")]
failures = counts.set_index("sample")[checks].T
failures.index = failures.index.str.removesuffix("_failures")
display(failures)

## One row from each sample

Take the first row with cells from each sample. They are examples, not typical events; a row can hold several jets.

$p_T$ is momentum across the beam (MeV); $\eta$ describes direction. Time is in ns.

In [ ]:
display(pd.DataFrame(examples)[["sample", "row_index", "jet_count", "cluster_count", "cell_count"]])
for example in examples:
    selected = ((jets["sample"] == example["sample"]) &
                (jets["source_file"] == example["source_file"]) &
                (jets["row_index"] == example["row_index"]))
    display(Markdown(f"**{example['sample']}: jets in this row**"))
    display(jets.loc[selected, ["jet_index", "jet_pt", "jet_eta", "jet_time_ATLAS",
                               "jet_time_spread", "jet_n_clusters"]].head(15))

## Jet plots

A histogram **bin** is an x-axis interval. Its height is the fraction of jets there. Each sample sums to one: compare shapes, not event rates. The $p_T$ axis is logarithmic.

In the timing plot, each point is a jet. Positive ATLAS time means late after the usual light-travel correction.

In [ ]:
figures = make_plots(jets, cells, OUTPUT)
display(Image(filename=figures["01_jet_kinematics"]))
display(Image(filename=figures["02_jet_timing"]))

## Cell plots

Each point is a **cell**: position along the beam ($z$, mm) versus time (ns).

These are all cells in one row, without separating jets. The tiny unpaired example cannot show a typical BIB pattern.

In [ ]:
display(Image(filename=figures["03_cell_timing"]))

## Missing values

`-999999` means missing, not measured. This first look applies no physics cuts and includes unmatched LLP-sample jets. Training uses stricter selections.

In [ ]:
quality = pd.DataFrame(summary["jet_value_quality"])
has_missing = (quality["nonfinite"] > 0) | (quality["minus_999999"] > 0)
display(quality.loc[has_missing, ["sample", "branch", "values", "nonfinite", "minus_999999"]])
print("Full checks and plots:", OUTPUT)

Next: [XGBoost](03_xgboost_baseline.ipynb). Later cell models: [analysis guide](../analysis/README.md).